# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MbaliThusi/Mbali-Flyrank-intern-rep./blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd, numpy as np, os
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])

d = df.replace([np.inf, -np.inf], np.nan).copy()
d["ctr_30"] = d["clicks_last_30d"] / d["impressions_last_30d"].replace(0, np.nan)
d["pos_bucket"] = pd.cut(d["avg_position"], [0, 3, 10, 20, 1000], labels=["1-3", "4-10", "11-20", "20+"])
ok = d["impressions_last_30d"] >= 100
bench_p75 = d[ok].groupby("pos_bucket", observed=True)["ctr_30"].quantile(.75)
bench_med = d[ok].groupby("pos_bucket", observed=True)["ctr_30"].median()

d["opp_refresh"] = np.where(d["clicks_prev_30d"] >= 10, (d["clicks_prev_30d"] - d["clicks_last_30d"]).clip(lower=0), 0)
gap = (d["pos_bucket"].map(bench_p75).astype(float) - d["ctr_30"]).clip(lower=0)
d["opp_snippet"] = np.where(d["avg_position"] <= 10, gap * d["impressions_last_30d"], 0)
gain = (float(bench_med["4-10"]) - d["ctr_30"]).clip(lower=0)
d["opp_push"] = np.where((d["avg_position"] > 10) & (d["avg_position"] <= 20), 0.5 * gain * d["impressions_last_30d"], 0)

opps = d[["opp_refresh", "opp_snippet", "opp_push"]].fillna(0)
code_map = {"opp_refresh": "REFRESH_DECLINING", "opp_snippet": "FIX_SNIPPET", "opp_push": "PUSH_RANKING"}
d["score"] = opps.max(axis=1).round(1)
d["reason_code"] = opps.idxmax(axis=1).map(code_map)
low = (d["impressions_last_30d"] < 100) & (d["clicks_prev_30d"] < 10)
d.loc[low | (d["score"] == 0), ["reason_code", "score"]] = ["LOW_DATA", 0.0]
d["confidence"] = np.select(
    [(d["impressions_last_30d"] >= 1000) & (d["clicks_prev_30d"] >= 50), (d["impressions_last_30d"] >= 300)],
    ["high", "medium"], default="low")

queue = d.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
print(len(queue), "rows ranked")

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv
30000 rows ranked


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
REASONS = {
    "REFRESH_DECLINING": "Clicks fell vs the previous 30 days; refresh to recover them",
    "FIX_SNIPPET": "Ranks top 10 but CTR is below peers at that position; rewrite title/meta",
    "PUSH_RANKING": "Ranks 10-20 with real impressions; improve content to reach page one",
    "LOW_DATA": "Too little traffic to judge; no action",
}
queue["why"] = queue["reason_code"].map(REASONS)
pd.set_option("display.max_colwidth", 70, "display.width", 220)
print(queue[["rank", "content_id", "reason_code", "score", "confidence", "why"]].head(10).to_string(index=False))
print("\nactions in top 200:\n", queue.head(200)["reason_code"].value_counts())

 rank           content_id       reason_code  score confidence                                                                      why
    1 content_8451fc6f034d       FIX_SNIPPET 1315.7     medium Ranks top 10 but CTR is below peers at that position; rewrite title/meta
    2 content_4a6607efcb46       FIX_SNIPPET  968.4     medium Ranks top 10 but CTR is below peers at that position; rewrite title/meta
    3 content_db5989a78dd3       FIX_SNIPPET  841.6       high Ranks top 10 but CTR is below peers at that position; rewrite title/meta
    4 content_07e0b9af8b1a REFRESH_DECLINING  678.0       high             Clicks fell vs the previous 30 days; refresh to recover them
    5 content_36ff89c8214e       FIX_SNIPPET  566.5       high Ranks top 10 but CTR is below peers at that position; rewrite title/meta
    6 content_aaef01a50def       FIX_SNIPPET  523.9       high Ranks top 10 but CTR is below peers at that position; rewrite title/meta
    7 content_c84a0ab98e90       FIX_SNIPPET  52

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
print("youngest content (days):", df["content_age_days"].min(), "| oldest:", df["content_age_days"].max())
print("share of rows with LOW_DATA:", (queue["reason_code"] == "LOW_DATA").mean().round(3))
print("share of top-100 rated low confidence:", (queue.head(100)["confidence"] == "low").mean().round(3))

youngest content (days): 90 | oldest: 564
share of rows with LOW_DATA: 0.708
share of top-100 rated low confidence: 0.0


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:
checks = {
    "REFRESH_DECLINING": "Is the drop seasonal or noise? Was the page already updated?",
    "FIX_SNIPPET": "Is the low CTR due to branded queries or SERP features?",
    "PUSH_RANKING": "Is page-one competition realistic for this topic?",
}
review = queue.head(20).copy()
review["human_check"] = review["reason_code"].map(checks)
print(review[["rank", "content_id", "reason_code", "confidence", "human_check"]].to_string(index=False))

 rank           content_id       reason_code confidence                                                  human_check
    1 content_8451fc6f034d       FIX_SNIPPET     medium      Is the low CTR due to branded queries or SERP features?
    2 content_4a6607efcb46       FIX_SNIPPET     medium      Is the low CTR due to branded queries or SERP features?
    3 content_db5989a78dd3       FIX_SNIPPET       high      Is the low CTR due to branded queries or SERP features?
    4 content_07e0b9af8b1a REFRESH_DECLINING       high Is the drop seasonal or noise? Was the page already updated?
    5 content_36ff89c8214e       FIX_SNIPPET       high      Is the low CTR due to branded queries or SERP features?
    6 content_aaef01a50def       FIX_SNIPPET       high      Is the low CTR due to branded queries or SERP features?
    7 content_c84a0ab98e90       FIX_SNIPPET     medium      Is the low CTR due to branded queries or SERP features?
    8 content_8c19996aa890       FIX_SNIPPET       high      Is 

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
# Triggers: write these thresholds in the markdown cell
print("current decline rate:", round(float((d["clicks_last_30d"] < d["clicks_prev_30d"]).mean()), 3), "(review if it moves >10 points)")
print("current median CTR:", round(float(d["ctr_30"].median()), 4), "(review if it moves >25% relative)")
print("current share high confidence:", round(float((queue["confidence"] == "high").mean()), 3))
print("benchmark CTR by position (recompute monthly):\n", bench_p75.round(4))

current decline rate: 0.227 (review if it moves >10 points)
current median CTR: 0.0 (review if it moves >25% relative)
current share high confidence: 0.021
benchmark CTR by position (recompute monthly):
 pos_bucket
1-3      0.0080
4-10     0.0056
11-20    0.0051
20+      0.0026
Name: ctr_30, dtype: float64


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
os.makedirs("work/outputs", exist_ok=True)
cols = ["rank", "client_id", "content_id", "reason_code", "score", "confidence",
        "clicks_prev_30d", "clicks_last_30d", "impressions_last_30d", "avg_position", "days_since_last_update"]
queue[cols].to_csv("work/outputs/content_action_queue.csv", index=False)
bench_p75.rename("p75_ctr").to_csv("work/outputs/ctr_benchmark_by_position.csv")
queue.head(100)["reason_code"].value_counts().to_csv("work/outputs/top100_action_mix.csv")
print(os.listdir("work/outputs"))

['ctr_benchmark_by_position.csv', 'top100_action_mix.csv', 'content_action_queue.csv']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.